# Exploratory Data Analysis (EDA)
## Personalized E-Commerce Recommendation System
**Dataset:** Amazon Reviews 2023 (Curated by McAuley Lab, University of California San Diego)
**Categories:** All Beauty, Appliances, Digital Music

---

### Objectives of this Analysis:
1. Validate dataset shape, feature schema, and data types.
2. Investigate missing values, price distributions, and imputation results.
3. Quantify user interaction behavior, Pareto power-law skewness, and sparsity.
4. Analyze product popularity and the long-tail catalog phenomenon.
5. Provide actionable insights for **Collaborative Filtering**, **Content-Based Filtering**, and **Cold-Start** handling.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Global aesthetics
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.size"] = 11

DATA_DIR = Path("../data/processed")


## 1. Dataset Ingestion & Schema Inspection

In [ ]:
df_products = pd.read_csv(DATA_DIR / "products.csv")
df_interactions = pd.read_csv(DATA_DIR / "interactions.csv")

print(f"Products Table Shape: {df_products.shape[0]:,} rows x {df_products.shape[1]} columns")
print(f"Interactions Table Shape: {df_interactions.shape[0]:,} rows x {df_interactions.shape[1]} columns")


In [ ]:
df_products.info()
df_products.head(3)


In [ ]:
df_interactions.info()
df_interactions.head(3)


## 2. Missing Value & Duplicate Analysis

In [ ]:
print("=== Missing Values in Products Table ===")
print(df_products.isnull().sum())
print("\n=== Missing Values in Interactions Table ===")
print(df_interactions.isnull().sum())

print(f"\nDuplicate Product IDs: {df_products['product_id'].duplicated().sum()}")
print(f"Duplicate (User, Product) Interaction Pairs: {df_interactions.duplicated(subset=['user_id', 'product_id']).sum()}")


## 3. High-Level Summary Statistics & Matrix Sparsity
Matrix sparsity is defined as:
$$\text{Sparsity} = \left(1 - \frac{|Interactions|}{|Users| \times |Items|}\right) \times 100\%$$


In [ ]:
n_users = df_interactions['user_id'].nunique()
n_products = df_products['product_id'].nunique()
n_interactions = len(df_interactions)

sparsity = (1.0 - (n_interactions / (n_users * n_products))) * 100.0

print(f"Unique Users: {n_users:,}")
print(f"Unique Products: {n_products:,}")
print(f"Total Interactions: {n_interactions:,}")
print(f"Interaction Matrix Sparsity: {sparsity:.4f}%")


## 4. Rating Distribution Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
rating_counts = df_interactions['rating'].value_counts().sort_index()
sns.barplot(x=rating_counts.index, y=rating_counts.values, palette="mako", ax=ax)
ax.set_title("Interaction Rating Distribution (1.0 to 5.0 Stars)", fontweight="bold")
ax.set_xlabel("Star Rating")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, xytext=(0, 4), textcoords='offset points')
plt.tight_layout()
plt.show()


## 5. Product Category Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
cat_counts = df_products['category'].value_counts()
sns.barplot(x=cat_counts.index, y=cat_counts.values, palette=["#4f46e5", "#06b6d4", "#10b981"], ax=ax)
ax.set_title("Catalog Product Count by Category", fontweight="bold")
ax.set_xlabel("Category")
ax.set_ylabel("Products Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, xytext=(0, 4), textcoords='offset points')
plt.tight_layout()
plt.show()


## 6. User Activity & Interaction Frequency (Pareto Skew)

In [ ]:
user_interactions = df_interactions['user_id'].value_counts()
print(f"Median interactions per user: {user_interactions.median():.1f}")
print(f"Max interactions by a single user: {user_interactions.max()}")
print(f"Users with >= 2 interactions: {(user_interactions >= 2).sum():,} ({(user_interactions >= 2).mean()*100:.2f}%)")
print(f"Users with >= 3 interactions: {(user_interactions >= 3).sum():,} ({(user_interactions >= 3).mean()*100:.2f}%)")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(user_interactions.values, bins=50, color="#6366f1", log=True, edgecolor="#1e1b4b")
ax.set_title("Histogram of User Interactions (Log Scale)", fontweight="bold")
ax.set_xlabel("Interactions per User")
ax.set_ylabel("User Count (Log Scale)")
plt.tight_layout()
plt.show()


## 7. Product Popularity & Long-Tail Catalog Analysis

In [ ]:
prod_interactions = df_interactions['product_id'].value_counts()
print("Top 10 Most Interacted Products:")
top10_ids = prod_interactions.head(10).index
top10_details = df_products[df_products['product_id'].isin(top10_ids)][['product_id', 'title', 'category', 'price', 'rating']]
top10_details['interaction_count'] = top10_details['product_id'].map(prod_interactions)
display(top10_details.sort_values('interaction_count', ascending=False))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
counts_sorted = prod_interactions.values
ax.plot(range(1, len(counts_sorted) + 1), counts_sorted, color="#ec4899", lw=2)
ax.fill_between(range(1, len(counts_sorted) + 1), counts_sorted, color="#fbcfe8", alpha=0.6)

cum_interactions = np.cumsum(counts_sorted)
top_20_pct = int(len(counts_sorted) * 0.20)
share_20 = (cum_interactions[top_20_pct] / cum_interactions[-1]) * 100

ax.axvline(x=top_20_pct, color="#831843", linestyle="--", label=f"Top 20% Items ({share_20:.1f}% Interactions)")
ax.set_title("Long-Tail Distribution of Product Popularity", fontweight="bold")
ax.set_xlabel("Product Rank")
ax.set_ylabel("Interaction Count")
ax.legend(frameon=True)
plt.tight_layout()
plt.show()


## 8. High-Rated Products with Minimum Interaction Threshold
To prevent items with only a single 5-star rating from dominating recommendations, a minimum interaction threshold is applied.


In [ ]:
min_threshold = 5
popular_prods = df_products[df_products['interaction_count'] >= min_threshold].copy()
popular_prods = popular_prods.sort_values(by=['rating', 'interaction_count'], ascending=[False, False])

print(f"Products with at least {min_threshold} interactions: {len(popular_prods)}")
display(popular_prods[['title', 'category', 'price', 'rating', 'interaction_count']].head(10))


## 9. Key Findings for Algorithmic Design
1. **High Matrix Sparsity (>99.9%):**
   - Direct item-to-item or user-to-user Collaborative Filtering faces sparse neighborhood overlaps.
   - **Resolution:** Truncated SVD dimensionality reduction (rank $k=30$) compresses the sparse user-item interaction matrix into dense latent spaces, eliminating zero-overlap bottlenecks.
2. **Extreme Power-Law Skew in User Activity:**
   - Most users have 1-2 interactions, while power users have dozens.
   - **Resolution:** Content-Based Filtering and onboarding preferences are vital for cold-start and low-activity users.
3. **Multi-Category Catalog Breadth:**
   - The catalog features distinct product categories ("All Beauty", "Appliances", "Digital Music"), enabling user category preference filtering and cross-category discovery.
4. **Hybrid Engine Synergy:**
   - $\alpha \times \text{CF} + (1 - \alpha) \times \text{CB}$ combines latent collaborative signals with semantic metadata signals, providing reliable recommendations across all user activity levels.
